# TT-07 — Gradient Boosting: Dự đoán thu nhập
Adult Census Income — hỗ trợ chấm điểm hồ sơ vay tiêu dùng.

**Bẫy dữ liệu:** `' ?'` có dấu cách trước, chuỗi có khoảng trắng thừa, nhãn test có dấu chấm cuối
(`'>50K.'`), `education`/`education-num` trùng lặp, `fnlwgt` không liên quan cá nhân.

**Nguyên tắc phương pháp trong notebook này:** tập test (`adult.test`, 16.281 dòng) chỉ được
dùng **đúng một lần**, ở bước đánh giá cuối cùng. Mọi lựa chọn siêu tham số (grid learning_rate ×
n_estimators, đường loss để chẩn đoán overfit) đều thực hiện trên tập train — bằng cross-validation
hoặc một tập validation tách riêng khỏi train — **không bao giờ nhìn vào test trước khi chốt mô hình**.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, AdaBoostClassifier)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score, log_loss,
                              accuracy_score, confusion_matrix, ConfusionMatrixDisplay)
import joblib

COLS = ['age','workclass','fnlwgt','education','education-num','marital-status',
        'occupation','relationship','race','sex','capital-gain','capital-loss',
        'hours-per-week','native-country','income']

RANDOM_STATE = 42

## 1. Đọc & làm sạch dữ liệu
Dùng đúng bộ train/test chính thức UCI (`adult.data` = 32.561 dòng, `adult.test` = 16.281 dòng) thay vì tự split, để số liệu so sánh được với mức tham chiếu công khai.

In [ ]:
def find_data_dir():
    candidates = [Path('../data'), Path('data'), Path('../../data')]
    for c in candidates:
        if (c / 'adult.data').exists():
            return c
    raise FileNotFoundError(
        "Không tìm thấy adult.data. Đặt file trong thư mục 'data/' cùng cấp với "
        "'notebooks/' (xem README mục 'Dữ liệu'), hoặc sửa đường dẫn thủ công.")

DATA_DIR = find_data_dir()
print('Đang dùng thư mục dữ liệu (đường dẫn tương đối):', DATA_DIR)

def load_clean(path, skiprows=0):
    # Xử lý 4 bẫy kinh điển của Adult Census Income:
    # 1) ' ?' có dấu cách trước -> na_values='?' + skipinitialspace=True
    # 2) khoảng trắng thừa quanh mọi giá trị chuỗi -> .str.strip()
    # 3) file adult.test có dòng header rác đầu tiên và nhãn kết thúc bằng dấu
    #    chấm ('>50K.' thay vì '>50K') -> skiprows=1 + .str.rstrip('.')
    # 4) education (chuỗi) và education-num (số) là cùng một thông tin,
    #    fnlwgt là trọng số điều tra dân số không liên quan cá nhân -> bỏ cả hai
    df = pd.read_csv(path, header=None, names=COLS, skiprows=skiprows,
                      skipinitialspace=True, na_values='?')
    for c in df.select_dtypes(include='object').columns:
        df[c] = df[c].str.strip()
    df['income'] = df['income'].str.rstrip('.')
    df['income'] = (df['income'] == '>50K').astype(int)
    df = df.drop(columns=['fnlwgt', 'education'])
    return df

train_df = load_clean(DATA_DIR / 'adult.data')
test_df = load_clean(DATA_DIR / 'adult.test', skiprows=1)
print('Train:', train_df.shape, '| Test:', test_df.shape)
print('Tỉ lệ >50K (train):', round(train_df['income'].mean(), 3))
train_df.isna().sum()[train_df.isna().sum() > 0]

In [ ]:
y_train, y_test = train_df.pop('income'), test_df.pop('income')
X_train, X_test = train_df, test_df
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
num_cols = X_train.select_dtypes(exclude='object').columns.tolist()
cat_cols, num_cols

## 2. EDA nhanh

In [ ]:
eda = X_train.assign(income=y_train)
display(eda.groupby('education-num')['income'].mean().round(3))
display(eda.groupby('marital-status')['income'].mean().round(3))
print('capital-gain = 0:', f"{(X_train['capital-gain']==0).mean():.1%}")

## 3. Pipeline tiền xử lý
Boosting **không cần scale** số, chỉ OneHot cho biến phân loại.

In [ ]:
pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols),
])
def make_pipe(model):
    return Pipeline([('pre', pre), ('clf', model)])

## 4. Tách validation từ TRAIN (không đụng vào test)

Để dò siêu tham số và vẽ đường loss mà không rò rỉ thông tin từ test, ta cắt riêng 20% của
`X_train` làm tập validation (`X_val`). Tập `X_test` chỉ xuất hiện lại ở **bước 9 — đánh giá cuối cùng**.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)
print('X_tr:', X_tr.shape, '| X_val:', X_val.shape)

## 5. Baseline: Dummy & Decision Tree
Không có siêu tham số nào được dò trên baseline này nên đánh giá thẳng trên test là hợp lệ (không có bước tuning nào phía trước để rò rỉ).

In [ ]:
results = {}
for name, model in [('Dummy', DummyClassifier(strategy='most_frequent')),
                     ('DecisionTree', DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))]:
    pipe = make_pipe(model).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    results[name] = {'pr_auc': average_precision_score(y_test, proba),
                      'roc_auc': roc_auc_score(y_test, proba)}
    print(f"{name}: PR-AUC={results[name]['pr_auc']:.3f}, ROC-AUC={results[name]['roc_auc']:.3f}")

## 6. Dò lưới learning_rate × n_estimators bằng CROSS-VALIDATION (chỉ trên train)

**Sửa lỗi phương pháp quan trọng nhất của bản trước:** lưới siêu tham số trước đây chấm điểm
bằng `roc_auc_score(y_test, ...)` — tức là chọn mô hình dựa trên chính tập sẽ dùng để báo cáo kết
quả cuối, làm ROC-AUC lạc quan hơn thực tế. Ở đây, lưới được chấm bằng `cross_val_score` với
`StratifiedKFold(3)` **chỉ trên `X_train`/`y_train`** — test không tham gia bước chọn mô hình.

> `learning_rate` nhỏ + `n_estimators` lớn nhìn chung ổn định hơn (ít nhạy với việc dừng sớm/muộn),
> trong khi `learning_rate` lớn hội tụ nhanh nhưng dễ đạt đỉnh sớm rồi giảm nhẹ khi thêm cây
> (bắt đầu overfit) — quan hệ NGHỊCH giữa hai tham số.

In [ ]:
lrs = [0.01, 0.05, 0.3]
n_ests = [100, 300, 500]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

t0 = time.time()
grid = np.zeros((len(lrs), len(n_ests)))
for i, lr in enumerate(lrs):
    for j, ne in enumerate(n_ests):
        m = make_pipe(GradientBoostingClassifier(
            n_estimators=ne, learning_rate=lr, max_depth=3, subsample=0.8,
            random_state=RANDOM_STATE))
        scores = cross_val_score(m, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        grid[i, j] = scores.mean()
grid_search_time = time.time() - t0
print(f'Thời gian dò lưới (5x9 = 45 lượt fit CV): {grid_search_time:.1f}s')

grid_df = pd.DataFrame(grid, index=lrs, columns=n_ests)
grid_df.index.name = 'learning_rate'; grid_df.columns.name = 'n_estimators'
display(grid_df.round(4))

best_i, best_j = np.unravel_index(np.argmax(grid), grid.shape)
best_lr, best_ne = lrs[best_i], n_ests[best_j]
print(f'Siêu tham số tốt nhất theo CV (không nhìn test): learning_rate={best_lr}, n_estimators={best_ne}')

In [ ]:
plt.figure(figsize=(6, 4))
im = plt.imshow(grid, cmap='viridis')
plt.xticks(range(len(n_ests)), n_ests); plt.yticks(range(len(lrs)), lrs)
plt.xlabel('n_estimators'); plt.ylabel('learning_rate')
plt.title('ROC-AUC trung bình (3-fold CV trên train): learning_rate x n_estimators')
for i in range(len(lrs)):
    for j in range(len(n_ests)):
        plt.text(j, i, f'{grid[i,j]:.3f}', ha='center', va='center', color='white')
plt.colorbar(im); plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'lr_vs_nestimators.png'), dpi=120)
plt.show()

## 7. Gradient Boosting cuối cùng — huấn luyện trên toàn bộ train

Dùng siêu tham số vừa chọn bằng CV. `validation_fraction`/`n_iter_no_change` bên trong
`GradientBoostingClassifier` tự tách một phần nhỏ của **train** để dừng sớm nội bộ — đây **không
phải** là rò rỉ từ test, vì test vẫn chưa được động tới.

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=best_ne, learning_rate=best_lr, max_depth=3,
    subsample=0.8, validation_fraction=0.1, n_iter_no_change=20,
    random_state=RANDOM_STATE,
)
gb_pipe = make_pipe(gb)
t0 = time.time()
gb_pipe.fit(X_train, y_train)
gb_time = time.time() - t0
print(f'Đã huấn luyện GradientBoosting trong {gb_time:.2f}s, dừng sớm ở cây thứ {gb.n_estimators_}/{best_ne}')

## 8. Train/Validation loss theo số cây — chẩn đoán overfit đúng cách

Đường loss được vẽ trên **`X_tr`/`X_val`** (tách từ train ở bước 4), không phải test.

Hai kịch bản được so sánh:
- **(a) Cấu hình đã regularize** (giống mô hình cuối: `subsample=0.8`, `learning_rate` nhỏ) — dự
  kiến loss validation vẫn còn giảm ở cây cuối, tức là **trong phạm vi `n_estimators` đã chọn,
  mô hình CHƯA bộc lộ overfit rõ ràng** — hiểu đúng con số best_iteration gần cuối dải là "chưa kịp
  overfit" chứ không phải "bắt đầu overfit".
- **(b) Cấu hình cố ý KHÔNG regularize** (`learning_rate=0.3`, `subsample=1.0`, 1000 cây) — dùng để
  minh hoạ overfit THẬT xảy ra như thế nào: loss train tiếp tục giảm trong khi loss validation
  chạm đáy rồi tăng trở lại.

In [ ]:
# (a) cấu hình đã regularize, refit trên X_tr để có thể trace loss trên X_val
gb_reg = GradientBoostingClassifier(n_estimators=best_ne, learning_rate=best_lr, max_depth=3,
                                     subsample=0.8, random_state=RANDOM_STATE)
pipe_reg = make_pipe(gb_reg).fit(X_tr, y_tr)
Xtr_t = pipe_reg.named_steps['pre'].transform(X_tr)
Xval_t = pipe_reg.named_steps['pre'].transform(X_val)
fitted_reg = pipe_reg.named_steps['clf']
train_loss_reg = [log_loss(y_tr, p) for p in fitted_reg.staged_predict_proba(Xtr_t)]
val_loss_reg = [log_loss(y_val, p) for p in fitted_reg.staged_predict_proba(Xval_t)]
best_iter_reg = int(np.argmin(val_loss_reg))

# (b) cấu hình cố ý không regularize, để MINH HOẠ overfit thật
gb_over = GradientBoostingClassifier(n_estimators=1000, learning_rate=0.3, max_depth=3,
                                      subsample=1.0, random_state=RANDOM_STATE)
pipe_over = make_pipe(gb_over).fit(X_tr, y_tr)
Xtr_t2 = pipe_over.named_steps['pre'].transform(X_tr)
Xval_t2 = pipe_over.named_steps['pre'].transform(X_val)
fitted_over = pipe_over.named_steps['clf']
train_loss_over = [log_loss(y_tr, p) for p in fitted_over.staged_predict_proba(Xtr_t2)]
val_loss_over = [log_loss(y_val, p) for p in fitted_over.staged_predict_proba(Xval_t2)]
best_iter_over = int(np.argmin(val_loss_over))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(train_loss_reg, label='Train loss')
axes[0].plot(val_loss_reg, label='Validation loss')
axes[0].axvline(best_iter_reg, color='red', ls='--', label=f'Loss thấp nhất (cây {best_iter_reg})')
axes[0].set_title(f'(a) Đã regularize — chưa overfit trong {len(val_loss_reg)} cây')
axes[0].set_xlabel('Số cây (boosting stage)'); axes[0].set_ylabel('Log loss'); axes[0].legend()

axes[1].plot(train_loss_over, label='Train loss')
axes[1].plot(val_loss_over, label='Validation loss')
axes[1].axvline(best_iter_over, color='red', ls='--', label=f'Overfit bắt đầu (cây {best_iter_over})')
axes[1].set_title('(b) KHÔNG regularize — overfit rõ ràng')
axes[1].set_xlabel('Số cây (boosting stage)'); axes[1].set_ylabel('Log loss'); axes[1].legend()

plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'loss_theo_so_cay.png'), dpi=120)
plt.show()

print(f'(a) Regularized: loss validation thấp nhất ở cây {best_iter_reg}/{len(val_loss_reg)} '
      f'-> vẫn còn giảm tới gần cuối, CHƯA quan sát được overfit trong phạm vi đã chọn.')
print(f'(b) Không regularize: loss validation thấp nhất ở cây {best_iter_over}/{len(val_loss_over)}, '
      f'sau đó tăng từ {min(val_loss_over):.3f} lên {val_loss_over[-1]:.3f} -> overfit thật.')

## 9. Đánh giá cuối cùng trên TEST (chỉ một lần, sau khi đã chốt mọi lựa chọn ở trên)

In [ ]:
proba_gb = gb_pipe.predict_proba(X_test)[:, 1]
results['GradientBoosting'] = {'pr_auc': average_precision_score(y_test, proba_gb),
                                'roc_auc': roc_auc_score(y_test, proba_gb)}
print(f"GradientBoosting trên TEST: ROC-AUC={results['GradientBoosting']['roc_auc']:.3f}, "
      f"PR-AUC={results['GradientBoosting']['pr_auc']:.3f}")

## 10. So sánh Bagging (Random Forest) vs Boosting vs AdaBoost
Ba mô hình dùng siêu tham số mặc định/hợp lý, không có bước dò trên test nào cho riêng bước so sánh này.

In [ ]:
models = {
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'GradientBoosting': gb,
    'AdaBoost': AdaBoostClassifier(n_estimators=300, random_state=RANDOM_STATE),
}
comp_rows = []
for name, model in models.items():
    pipe = make_pipe(model)
    t0 = time.time(); pipe.fit(X_train, y_train); dt = time.time() - t0
    p = pipe.predict_proba(X_test)[:, 1]
    comp_rows.append({'model': name,
                       'pr_auc': round(average_precision_score(y_test, p), 3),
                       'roc_auc': round(roc_auc_score(y_test, p), 3),
                       'train_time_s': round(dt, 2)})
comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(str(DATA_DIR.parent / 'reports' / 'model_comparison.csv'), index=False)
comp_df

## 11. GradientBoosting vs HistGradientBoosting (tốc độ)
Dùng cùng `n_estimators`/`learning_rate` đã chọn ở bước 6, đo trong cùng một lần chạy notebook để số liệu nhất quán với README.

In [ ]:
pre_dense = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', 'passthrough', num_cols),
])
hgb = HistGradientBoostingClassifier(max_iter=best_ne, learning_rate=best_lr,
                                      early_stopping=True, random_state=RANDOM_STATE)
hgb_pipe = Pipeline([('pre', pre_dense), ('clf', hgb)])
t0 = time.time(); hgb_pipe.fit(X_train, y_train); hgb_time = time.time() - t0
p_hgb = hgb_pipe.predict_proba(X_test)[:, 1]
speedup = gb_time / hgb_time
print(f'GradientBoosting: {gb_time:.2f}s | HistGradientBoosting: {hgb_time:.2f}s (nhanh hơn {speedup:.1f}x)')
print(f'ROC-AUC HGB = {roc_auc_score(y_test, p_hgb):.3f}')

## 12. Gắn kết quả với nghiệp vụ: ma trận nhầm lẫn & chọn ngưỡng theo chi phí

Bài toán gốc là **chấm điểm hồ sơ vay tiêu dùng**: `income` dự đoán là một đầu vào để quyết định
duyệt/từ chối. Ngưỡng xác suất mặc định 0.5 không tính đến việc hai loại sai lầm có **chi phí khác
nhau**:

- **False Positive** (dự đoán >50K nhưng thực tế không) → duyệt nhầm khách có khả năng tài chính
  yếu hơn thực tế → rủi ro nợ xấu.
- **False Negative** (dự đoán ≤50K nhưng thực tế >50K) → từ chối nhầm khách hàng tốt → mất doanh
  thu lãi vay, mất khách hàng.

Giả định minh hoạ (cần công ty tài chính cung cấp số thật): từ chối nhầm khách tốt tốn kém gấp
~3 lần so với duyệt nhầm một khách yếu hơn dự kiến (`C_FN=3`, `C_FP=1`), vì công ty còn các lớp
kiểm soát rủi ro khác (tài sản đảm bảo, hạn mức thấp ban đầu) để bù cho FP, trong khi FN là mất
khách vĩnh viễn. Ta dò ngưỡng tối thiểu hoá `chi phí = C_FP × FP + C_FN × FN` trên TEST — đây là
bước *áp dụng* mô hình đã chốt, không phải bước *chọn* mô hình, nên dùng test ở đây là hợp lệ.

In [ ]:
C_FP, C_FN = 1, 3
thresholds = np.linspace(0.01, 0.99, 99)
costs = [C_FP * confusion_matrix(y_test, (proba_gb >= th).astype(int)).ravel()[1]
         + C_FN * confusion_matrix(y_test, (proba_gb >= th).astype(int)).ravel()[2]
         for th in thresholds]
best_th = float(thresholds[int(np.argmin(costs))])
print(f'Ngưỡng tối ưu theo chi phí: {best_th:.2f} (chi phí = {min(costs)}, so với ngưỡng 0.5)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(thresholds, costs)
axes[0].axvline(best_th, color='red', ls='--', label=f'Ngưỡng tối ưu={best_th:.2f}')
axes[0].axvline(0.5, color='gray', ls=':', label='Ngưỡng mặc định=0.5')
axes[0].set_xlabel('Ngưỡng xác suất'); axes[0].set_ylabel(f'Chi phí (C_FP={C_FP}, C_FN={C_FN})')
axes[0].set_title('Chi phí theo ngưỡng'); axes[0].legend()

ConfusionMatrixDisplay.from_predictions(y_test, (proba_gb >= 0.5).astype(int), ax=axes[1],
                                         colorbar=False, cmap='Blues')
axes[1].set_title('Ma trận nhầm lẫn @ ngưỡng 0.5')

ConfusionMatrixDisplay.from_predictions(y_test, (proba_gb >= best_th).astype(int), ax=axes[2],
                                         colorbar=False, cmap='Blues')
axes[2].set_title(f'Ma trận nhầm lẫn @ ngưỡng tối ưu {best_th:.2f}')

plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'threshold_cost.png'), dpi=120)
plt.show()

## 13. Kiểm tra thiên lệch theo `sex` và `race`
> Dữ liệu điều tra dân số Mỹ 1994 chứa định kiến lịch sử. Model có thể **học và khuếch đại** chênh
> lệch này. Không dùng cho quyết định thật về con người.

In [ ]:
pred = gb_pipe.predict(X_test)
bias_rows = []
for group_col in ['sex', 'race']:
    for g in X_test[group_col].unique():
        mask = X_test[group_col] == g
        if mask.sum() < 20:
            continue
        bias_rows.append({
            'group_col': group_col, 'group': g, 'n': int(mask.sum()),
            'positive_rate_true': round(y_test[mask].mean(), 3),
            'positive_rate_pred': round(pred[mask].mean(), 3),
            'accuracy': round(accuracy_score(y_test[mask], pred[mask]), 3),
        })
bias_df = pd.DataFrame(bias_rows)
bias_df.to_csv(str(DATA_DIR.parent / 'reports' / 'bias_by_group.csv'), index=False)
bias_df

In [ ]:
plt.figure(figsize=(7, 4))
sub = bias_df[bias_df.group_col == 'sex']
x = np.arange(len(sub))
plt.bar(x - 0.2, sub['positive_rate_pred'], width=0.4, label='Dự đoán >50K')
plt.bar(x + 0.2, sub['positive_rate_true'], width=0.4, label='Thực tế >50K')
plt.xticks(x, sub['group'])
plt.ylabel('Tỉ lệ >50K'); plt.title('Thiên lệch theo giới tính'); plt.legend()
plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'bias_by_group.png'), dpi=120)
plt.show()

**Nhận xét:** nam giới được mô hình dự đoán thu nhập >50K với tỉ lệ cao hơn nhiều so với nữ
giới, và người da trắng cao hơn nhiều so với người da đen — phản ánh đúng chênh lệch đã có trong
dữ liệu gốc (không phải mô hình "tự bịa ra" thiên lệch). Vì `sex`/`race` là biến đầu vào, gỡ bỏ hai
cột này không loại bỏ hoàn toàn thiên lệch: các biến khác như `occupation`, `relationship`,
`marital-status` có tương quan gián tiếp với `sex`/`race` (biến thay thế — *proxy variable*), nên
mô hình vẫn có thể tái tạo lại phần lớn chênh lệch. Accuracy cao ở một nhóm không đồng nghĩa với
công bằng nếu positive_rate dự đoán bị lệch xa so với positive_rate thực tế của nhóm đó.

## 14. Lưu model & tổng kết

⚠️ Thư mục `data/` (chứa `adult.data`, `adult.test`, ~6MB) **không nên commit vào git** — xem
`.gitignore` trong README. Model và báo cáo dưới đây được ghi vào `models/` và `reports/` ở gốc dự
án (`../models`, `../reports` tính từ `notebooks/`).

In [ ]:
MODEL_PATH = DATA_DIR.parent / 'models' / 'gb_pipeline.joblib'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(gb_pipe, MODEL_PATH)
print('Đã lưu model tại (đường dẫn tương đối):', MODEL_PATH)

summary = pd.DataFrame(results).T  # columns: pr_auc, roc_auc
summary['train_time_s'] = [np.nan, np.nan, round(gb_time, 2)]
summary.loc['HistGradientBoosting'] = [
    average_precision_score(y_test, p_hgb), roc_auc_score(y_test, p_hgb), round(hgb_time, 2)]
summary.to_csv(str(DATA_DIR.parent / 'reports' / 'final_summary.csv'))
summary